# 05 - Other TWPAs

This notebook builds two other designs with the same tools:

1. **A 4WM KI-TWPA with resonant phase matching and a modulated line**, from M. Malnou, *Synthesis of artificial transmission lines tailored for traveling-wave parametric processes*, Phys. Rev. Applied **25**, 054016 (2026). A resonant phase-matching (RPM) filter in one cell per supercell creates a notch whose dispersion compensates the Kerr phase mismatch of the pump, and a periodic modulation of the shunt capacitors opens a stopband that suppresses the pump third harmonic. The RPM cell is a user model, which shows how to make a custom cell work with the `bloch` model.
2. **A Floquet-mode TWPA**: the nonlinearity is switched on and off smoothly along the line, so that less signal is lost to frequency-conversion sidebands (K. Peng *et al.*, *Floquet-Mode Traveling-Wave Parametric Amplifiers*, PRX Quantum **3**, 020306 (2022)). It is a segmented device: a chain of uniform segments.

Both raise `passes` above its default, and `anderson_depth` with it (`passes - 10`).

In [ ]:
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np

from twpasolver import ModeSet, TWPAnalysis
from twpasolver.cmes.solver_config import BlochSolverConfig
from twpasolver.logger import log
from twpasolver.matrices_arrays import ABCDArray
from twpasolver.models import PORT_OUT, TWPA, LCLfBaseCell
from twpasolver.twoport import TwoPortModel

log.setLevel("WARNING")
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## 1. A 4WM KI-TWPA with resonant phase matching

### The supercell

The line is a 50 $\Omega$ LC ladder of kinetic inductances $L_0 = 100$ pH with $I_* = 100\,\mu$A and no dc bias (pure 4WM). A supercell holds 9 unit cells (Appendix I of the paper):

* eight cells with shunt capacitors 51.3, 48.7, 42.0, 34.3, 29.3, 29.3, 34.3, 42.0 fF: two overlapped cosine modulations of the line impedance that open a stopband around 27 GHz;
* one RPM cell (shunt capacitor 48.7 fF) whose series inductor is shunted by a series LC branch ($L_r = 4.92$ nH, $C_r = 53.5$ fF).

The first eight are `LCLfBaseCell`s without finger inductance. The RPM cell is a user model: a `name`, a `single_abcd` method and `bloch_nl_elements()`, which tells the `bloch` model that the nonlinear inductor $L_0$ carries the cell's output current (accurate away from the notch, where the LC branch is a high impedance). `ktwpa(modulated=False)` gives every shunt capacitor the mean value, for the comparison below.

In [ ]:
class RPMCell(TwoPortModel):
    """Shunt C, then a series inductor L in parallel with a series LC branch (RPM filter)."""

    name: Literal["RPMCell"] = "RPMCell"
    L: float
    C: float
    Lr: float
    Cr: float

    def single_abcd(self, freqs):
        w = 2 * np.pi * np.asarray(freqs, dtype=float)
        Z_branch = 1j * w * self.Lr + 1 / (1j * w * self.Cr)
        Zs = 1 / (1 / (1j * w * self.L) + 1 / Z_branch)
        Y = 1j * w * self.C
        abcd = np.empty((len(w), 2, 2), dtype=complex)
        abcd[:, 0, 0] = 1
        abcd[:, 0, 1] = Zs
        abcd[:, 1, 0] = Y
        abcd[:, 1, 1] = 1 + Y * Zs
        return ABCDArray(abcd)

    def bloch_nl_elements(self):
        return [(PORT_OUT, self.L)]


L0, Istar = 100e-12, 100e-6
caps = np.array([51.3, 48.7, 42.0, 34.3, 29.3, 29.3, 34.3, 42.0]) * 1e-15
C_rpm, Lr, Cr = 48.7e-15, 4.92e-9, 53.5e-15


def ktwpa(modulated: bool = True, Ip0: float = 22e-6) -> TWPA:
    shunt = caps if modulated else np.full(8, np.mean([*caps, C_rpm]))
    last_C = C_rpm if modulated else shunt[0]
    cells = [LCLfBaseCell(L=L0, C=c, Lf=0.0) for c in shunt]
    cells.append(RPMCell(L=L0, C=last_C, Lr=Lr, Cr=Cr))
    return TWPA(cells=cells, N=556, Istar=Istar, Idc=0.0, Ip0=Ip0)


an = TWPAnalysis(twpa=ktwpa(), freqs=(0.5, 45, 2e-3))
lin = an.linear
print(
    f"{an.twpa.N_tot} cells, Bloch impedance at 1 GHz: {lin.Zb[0, np.searchsorted(lin.freqs, 1.0)]:.1f} Ohm"
)
print("stopbands (GHz):", np.round(lin.stopbands, 2))
print(
    f"notch from (L0 + Lr) Cr: {1 / (2 * np.pi * np.sqrt((L0 + Lr) * Cr)) / 1e9:.2f} GHz"
)

### Linear response

The notch sits where the LC branch, capacitive below its own resonance, cancels the series inductor: an open in the
series branch at $1/(2\pi\sqrt{(L_0+L_r)C_r}) = 9.71$ GHz (the paper quotes 10 GHz). Just below it $k^*$ rises
steeply, which is the extra pump wavenumber the 4WM phase matching needs. The modulation stopband around 27 GHz
contains the pump third harmonic $3 f_p \approx 27.3$ GHz.

In [ ]:
fp, Ip0 = 9.1, 22e-6
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 6), sharex=True)
ax1.plot(lin.freqs, lin.S21_db)
ax2.plot(lin.freqs, lin.k_star)
for ax in (ax1, ax2):
    for lo, hi in lin.stopbands:
        ax.axvspan(lo, hi, color="gray", alpha=0.3)
    ax.axvline(fp, color="C3", ls=":")
    ax.axvline(3 * fp, color="C3", ls="--")
ax1.set_ylim(-20, 2)
ax1.set_ylabel("$|S_{21}|$ (dB)")
ax2.set_ylabel("$k^*$ (rad/cell)")
ax2.set_xlabel("Frequency (GHz)")
plt.show()

### Gain

We pump at $f_p = 9.1$ GHz with $I_p = 22\,\mu$A, as in the paper. Pump, signal and idler alone do not show what happens in this line: the pump third harmonic $3 f_p$ and the up-conversion products $2 f_p + f_s$ and $2 f_p + f_i$ fall in or next to the modulation stopband and exchange power with the other modes, so we include them from the start.

In [ ]:
EXT4 = ModeSet(
    ["p", "s", "i", "p3", "p2s", "p2i"],
    [["i", "p+p-s"], ["p3", "p+p+p"], ["p2s", "p+p+s"], ["p2i", "p+p+i"]],
)
signals = np.linspace(4, 14, 205)
cfg_rpm = BlochSolverConfig(passes=60, anderson_depth=50)
r = an.gain(signals, pump=fp, Is0=1e-8, model="bloch", modes=EXT4, bloch_config=cfg_rpm)
print(f"converged {r.converged.sum()}/{len(signals)}")
ax = r.plot()
plt.show()

The gain is about 17-19.5 dB over 4-7.5 GHz and, mirrored around the pump, over 10.5-14 GHz, close to the 20 dB from 4 to 8 GHz of the paper's harmonic-balance simulation. It falls where the signal or the idler approaches the notch, between 8 and 10 GHz. One signal near 7.4 GHz does not converge: there $2 f_p + f_s$ sits at the lower edge of the modulation stopband; such points are flagged in `GainResult.converged`.

### Without the stopband

In a KI-TWPA the line is nearly dispersionless far below its cutoff, so the pump third harmonic $3 f_p$ is almost phase matched and grows along the line. Without the modulation the up-conversion products are nearly phase matched too and the power cascades to ever higher modes: no truncated mode set follows it, and with the products of the set above the forward-backward iteration does not converge. To isolate the harmonic, we compare the design with the unmodulated line using pump, signal, idler and $3 f_p$ only.

In [ ]:
P3 = ModeSet(["p", "s", "i", "p3"], [["i", "p+p-s"], ["p3", "p+p+p"]])
runs = {}
for name, twpa in (
    ("modulated (design)", ktwpa()),
    ("no modulation", ktwpa(modulated=False)),
):
    a = TWPAnalysis(twpa=twpa, freqs=(0.5, 45, 2e-3))
    r = a.gain(
        signals,
        pump=fp,
        Is0=1e-8,
        model="bloch",
        modes=P3,
        bloch_config=cfg_rpm,
    )
    third = 20 * np.log10(
        np.abs(r.b_out[:, r.mode("p3")]) / np.abs(r.b_out[:, r.mode("p")])
    )
    runs[name] = r
    print(
        f"{name}: mean gain {np.mean(r.gain_db):.1f} dB, third harmonic / pump at the output "
        f"{np.mean(third):.1f} dB, converged {r.converged.sum()}/{len(signals)}"
    )
for name, r in runs.items():
    r.plot_currents(title=name)
plt.show()

The stopband suppresses the third harmonic at the output by more than 20 dB (-30 against -6.5 dB relative to the pump), in line with the 20-27 dB the paper reports. Without it the harmonic takes a large share of the pump power along the line (mode currents above) and the gain drops; the iteration does not converge for about one signal in ten, a sign of the strong harmonic generation.

## 2. A Floquet-mode TWPA

In a uniformly pumped TWPA the pump converts the signal not only into the idler but also into sidebands ($f_p \pm f_s$, $2 f_p \pm f_s$ and the corresponding idler terms). Photons left in those modes at the output are lost information, which lowers the quantum efficiency. Peng *et al.* showed that switching the nonlinearity on and off adiabatically along the line maps the input and output linear modes onto single Floquet modes of the pumped line, so that the sideband content at the output nearly vanishes.

A position-dependent device is a **segmented** `TWPA`: its `cells` are themselves `TWPA` objects, each a uniform segment with its own unit cells, number of supercells `N` and nonlinearity (`Istar`, `Idc`); the outer `TWPA` chains them and carries the pump current. Each segment must hold a whole number of supercells. The `bloch` model solves every segment on its own Bloch modes and joins them with exact junction conditions.

We cut the KI-TWPA holotype of notebooks 03 and 04 (3WM, pump at 15.8 GHz) into 40 segments and scale its nonlinearity ($\propto 1/I_*^2$) with a Gaussian profile that keeps 10 % at the ends. The linear circuit is the same in every segment, so the profile adds no reflections.

In [ ]:
FREQS = (0.05, 50, 2e-3)  # GHz; reaches the 3rd pump harmonic (47.4 GHz)
FP = 15.8
MODES = ModeSet.extended_3wm(n_pump_harmonics=2, n_frequency_conversion=2)
CFG = BlochSolverConfig(passes=60, anderson_depth=50)
base = TWPA.from_file("imskit_v3_holotype.json")

In [ ]:
n_seg, floor, sigma = 40, 0.1, 0.2
u = (np.arange(n_seg) + 0.5) / n_seg
profile = floor + (1 - floor) * np.exp(-((u - 0.5) ** 2) / (2 * sigma**2))


def floquet_line(Ip0: float) -> TWPA:
    segments = [
        base.model_copy(
            deep=True,
            update={"N": base.N // n_seg, "Istar": base.Istar / np.sqrt(p), "Ip0": Ip0},
        )
        for p in profile
    ]
    return TWPA(cells=segments, Z0_ref=base.Z0_ref, Ip0=Ip0)


plt.figure(figsize=(5, 2.5))
plt.step(u, profile, where="mid")
plt.xlabel("Position along the line")
plt.ylabel("Relative nonlinearity")
plt.show()

The weaker average nonlinearity is compensated by a stronger pump (0.181 mA instead of 0.1 mA), which gives the
same mean gain. We keep the pump harmonics and two orders of frequency conversion, solve with the `bloch` model and
count, at the output, the photon flux $\propto |I|^2/f$ in the conversion modes relative to signal plus idler.

In [ ]:
LOST = ("ps", "pi", "p2s", "p2i")
signals = np.linspace(4, 11, 75)
results = {}
for name, twpa in (
    ("uniform", base.model_copy(update={"Ip0": 1e-4})),
    ("Floquet", floquet_line(1.809e-4)),
):
    an = TWPAnalysis(twpa=twpa, freqs=FREQS)
    r = an.gain(
        signals, pump=FP, Is0=1e-9, model="bloch", modes=MODES, bloch_config=CFG
    )
    f = an.problem(MODES, {"p": FP, "s": signals}).freqs
    flux = np.abs(r.b_out) ** 2 / np.abs(f)
    lost = sum(flux[:, r.mode(m)] for m in LOST)
    useful = flux[:, r.mode("s")] + flux[:, r.mode("i")]
    results[name] = (r, lost / useful)
    print(
        f"{name}: mean gain {np.mean(r.gain_db):.1f} dB, lost / (signal + idler) photons "
        f"{10 * np.log10(np.mean(lost / useful)):.1f} dB, converged {r.converged.sum()}/{len(signals)}"
    )

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
for name, (r, ratio) in results.items():
    ax1.plot(r.signal_freqs, r.gain_db, label=name)
    ax2.plot(r.signal_freqs, 10 * np.log10(ratio), label=name)
ax1.set_ylabel("Gain (dB)")
ax2.set_ylabel("Lost / useful photons (dB)")
for ax in (ax1, ax2):
    ax.set_xlabel("Signal frequency (GHz)")
    ax.legend()
plt.tight_layout()
plt.show()

At equal mean gain the tapered line leaves about 16 times (12 dB) fewer photons in the conversion modes. The mode
currents along the line show why: in the uniform device the conversion modes keep growing up to the output, while
in the Floquet device they are populated in the middle and return to the signal and idler as the nonlinearity
switches off.

In [ ]:
for name, (r, _) in results.items():
    ax = r.plot_currents(title=f"{name}: mode currents, averaged over the signals")
plt.show()